# `RelationDocument.bulk_relate()` Demo

Creates multiple graph edges in a **single `INSERT RELATION INTO` round-trip**, instead of calling `create_relation()` once per edge.

The payoff is clear when you need to seed large social graphs, batch-import follow lists, or wire up any N-to-M relationship in one shot.

In [ ]:
from surrealengine import (
    Document, RelationDocument,
    StringField, IntField, DateTimeField,
    create_connection,
)

## 1. Define models

In [ ]:
class User(Document):
    name  = StringField(required=True)
    email = StringField()

    class Meta:
        collection = "users"


class Follows(RelationDocument):
    """User -> follows -> User"""
    since    = IntField()       # year the follow started
    strength = IntField()       # e.g. mutual=2, one-way=1

    class Meta:
        collection = "follows"

## 2. Connect and create tables

In [ ]:
conn = create_connection(
    url       = "ws://db:8000/rpc",
    namespace = "test_ns",
    database  = "test_db",
    username  = "root",
    password  = "secret",
    make_default = True,
)
await conn.connect()
print("Connected")

await User.create_table()
await Follows.create_table()
print("Tables ready")

## 3. Seed some users

In [ ]:
names = ["Alice", "Bob", "Carol", "Dave", "Eve"]
users = []
for name in names:
    u = User(name=name, email=f"{name.lower()}@example.com")
    await u.save()
    users.append(u)
    print(f"Created {name}: {u.id}")

## 4. `bulk_relate` — all 5 follows in one round-trip

Each dict must have `"in"` and `"out"` keys.  Values can be:
- a saved `Document` instance (most common)
- a `RecordID` object
- a raw id string like `"users:abc123"`

Any extra keys become edge attributes on the `Follows` relation.

In [ ]:
alice, bob, carol, dave, eve = users

edges = await Follows.bulk_relate([
    {"in": alice, "out": bob,   "since": 2020, "strength": 2},
    {"in": alice, "out": carol, "since": 2021, "strength": 1},
    {"in": bob,   "out": carol, "since": 2019, "strength": 2},
    {"in": carol, "out": dave,  "since": 2022, "strength": 1},
    {"in": dave,  "out": eve,   "since": 2023, "strength": 1},
])

print(f"Created {len(edges)} edges")
for e in edges:
    print(f"  {e.id}  since={e.since}  strength={e.strength}")

## 5. Compare with single `create_relation` (for reference)

`bulk_relate` is equivalent to calling `create_relation` N times but uses **one** `INSERT RELATION INTO` statement.

In [ ]:
# This creates one extra edge the traditional way
single_edge = await Follows.create_relation(eve, alice, since=2024, strength=1)
print(f"Single edge: {single_edge.id}")

## 6. Query the graph

The edges are normal `Follows` instances — query them like any other document.

In [ ]:
# All follows with strength >= 2 (mutual)
strong = await Follows.objects.filter(strength__gte=2).all()
for f in strong:
    print(f"Mutual follow edge: {f.id}")

In [ ]:
# Everyone Alice follows
alice_follows = await alice.resolve_relation("follows")
print("Alice follows:", [u.get("name") for u in alice_follows])

## 7. Sync variant

If you're in a non-async context (e.g. a sync Django view or a script), use `bulk_relate_sync`.

In [ ]:
# Sync version — identical API, no await
# (only run this cell if you're running the kernel synchronously)
#
# sync_edges = Follows.bulk_relate_sync([
#     {"in": alice, "out": bob, "since": 2020, "strength": 2},
# ])

## 8. Cleanup

In [ ]:
for u in users:
    await u.delete()
print("Cleaned up")